# Машинное обучение. Практическая работа

В этой практической работе четыре обязательные задачи.

Они помогут понять, что вы действительно усвоили материал модуля. 


Удачи!

## Цели практической работы

Научиться:
- обучать модели, основанные на деревьях решений;
- научиться оценивать и увеличивать их качество.

## Что входит в практическую работу

1. Загрузите датасет и ознакомьтесь с ним.
2. Подготовьте базовую модель дерева решений и измерьте её качество.
3. Подготовьте базовую модель случайного леса и измерьте её качество.
4. Увеличьте точность модели случайного леса на тестовых данных.
5. Проведите анализ влияния признаков на модель.

## Что оценивается

* Выполнены все четыре задачи. Для каждой:
 * в коде нет ручных перечислений, все действия автоматизированы;
 * результаты вычислений и применённых операций корректны;
 * ответы на вопросы, где требуется, корректны и обоснованы; 
 * код читабелен: переменным даны осмысленные названия, соблюдены отступы и правила расстановки пробелов; стилизация кода соответствует рекомендациям [PEP 8](https://pythonworld.ru/osnovy/pep-8-rukovodstvo-po-napisaniyu-koda-na-python.html).

* Репозиторий проекта оформлен корректно:
 * содержит осмысленные коммиты, содержащие конкретные реализованные фичи;
 * ветки названы согласно назначению;
 * файлы, не связанные с проектом, не хранятся в репозитории.


## Как отправить работу на проверку

Сдайте практическую работу этого модуля через систему контроля версий Git сервиса Skillbox GitLab. После загрузки работы на проверку напишите об этом в личном кабинете своему куратору.

## Обязательные задачи

### Описание датасета:
- `id`— идентификатор записи;
- `is_manufacturer_name`— признак производителя автомобиля;

- `region_*`— регион;
- `x0_*`— тип топлива;
- `manufacturer_*`— производитель;
- `short_model_*`— сокращённая модель автомобиля;
- `title_status_*`— статус;
- `transmission_*`— коробка передач;
- `state_*`— штат;
- `age_category_*`— возрастная категория автомобиля;

- `std_scaled_odometer`— количество пройденных миль (после стандартизации);
- `year_std`— год выпуска (после стандартизации);
- `lat_std`— широта (после стандартизации);
- `long_std`— долгота (после стандартизации);
- `odometer/price_std`— отношение стоимости к пробегу автомобиля (после стандартизации);
- `desc_len_std`— количество символов в тексте объявления о продаже (после стандартизации);
- `model_in_desc_std`— количество наименований модели автомобиля в тексте объявления о продаже (после стандартизации);
- `model_len_std`— длина наименования автомобиля (после стандартизации);
- `model_word_count_std`— количество слов в наименовании автомобиля (после стандартизации);
- `month_std`— номер месяца размещения объявления о продаже автомобиля (после стандартизации);
- `dayofweek_std`— день недели размещения объявления о продаже автомобиля (после стандартизации);
- `diff_years_std`— количество лет между годом производства автомобиля и годом размещения объявления о продаже автомобиля (после стандартизации);

- `price`— стоимость;
- `price_category`— категория цены.

1. Подготовка базовой модели

Обучите простую модель классификации с помощью DecisionTreeClassifier на данных из датасета vehicles_dataset_prepared.csv. Для этого сделайте шаги:

1. Обучите модель дерева решений с зафиксированным random_state на тренировочной выборке.
2. Сделайте предикт на тестовой выборке.
3. Замерьте точность на тестовой выборке и выведите матрицу ошибок. 
4. Удалите фичи с нулевыми весами по feature_importance из тренировочной и тестовой выборок.
5. Заново обучите модель и измерьте качество.

In [126]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, confusion_matrix 

In [127]:
df = pd.read_csv('data/vehicles_dataset_prepared.csv')

df_prepared = df.copy()
df_prepared = df_prepared.drop(['price', 'odometer/price_std'], axis=1)

x = df_prepared.drop(['price_category'], axis=1)
y = df_prepared['price_category']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=50
)

In [128]:
# Ваш код здесь 
clf = DecisionTreeClassifier(random_state=50)
clf.fit(x_train, y_train)

y_predict = clf.predict(x_test)

print(f'Initial accuracy score: {accuracy_score(y_test, y_predict):.4f}')
print('Initial confusion matrix:')
print(confusion_matrix(y_test, y_predict))

Initial accuracy score: 0.6784
Initial confusion matrix:
[[710  51 185]
 [ 42 682 251]
 [187 212 566]]


In [114]:
imp = clf.feature_importances_
zero_imp_feat = x_train.columns[imp == 0]

x_train_cleaned = x_train.drop(columns=zero_imp_feat)
x_test_cleaned = x_test.drop(columns=zero_imp_feat)

clf_cleaned = DecisionTreeClassifier(random_state=50)
clf_cleaned.fit(x_train_cleaned, y_train)

y_pred_cleaned = clf_cleaned.predict(x_test_cleaned)

print('Zero importances removed')
print(f'Accuracy score: {accuracy_score(y_test, y_pred_cleaned):.4f}')
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_cleaned))

Zero importances removed
Accuracy score: 0.6784
Confusion matrix:
[[709  50 187]
 [ 37 690 248]
 [181 225 559]]


In [68]:
#В зависимости от random_state качество может слегка меняться. 

2. Подготовка модели случайного леса

Обучите простую модель классификации с помощью RandomForestClassifier. Для этого на новых урезанных семплах тренировочной и тестовой выборок обучите модель случайного леса с зафиксированным random_state=50. 

In [125]:
# Ваш код здесь 
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(random_state=50)
rf_clf.fit(x_train_cleaned, y_train)

RandomForestClassifier(random_state=50)

2. Сделайте предикт, посчитайте точность модели и матрицу ошибок. Сравните с предыдущей моделью дерева решений. Есть ли случаи, когда модель из пункта 1 отрабатывает лучше, чем модель случайного леса?

In [118]:
# Ваш код здесь 
y_rf_predicted = rf_clf.predict(x_test_cleaned)

print('Random Forest Classifier results:')
print(f'Accuracy score: {accuracy_score(y_test, y_rf_predicted):.4f}')
print('Confusrion matrix:')
print(confusion_matrix(y_rf_predicted, y_test))

Random Forest Classifier results:
Accuracy score: 0.7592
Confusrion matrix:
[[776  13 152]
 [ 38 811 209]
 [132 151 604]]


3. Тюнинг модели случайного леса

Увеличьте точность модели на тестовом датасете RandomForestClassifier c помощью тюнинга параметров. 

Параметры, которые можно настраивать для увеличения точности:
```
    `bootstrap'
    'max_depth'
    'max_features'
    'min_samples_leaf'
    'min_samples_split'
    'random_state'
    'n_estimators'

```

С описанием каждого из параметров можно ознакомиться [в документации](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

Задание засчитывается, если значение метрики строго выше 0.76 на тестовом датасете.

In [143]:
from sklearn.model_selection import RandomizedSearchCV 

param_dist = {
    'n_estimators': [400, 500, 700, 1000],
    'max_depth': [None, 30],
    'min_samples_split': [2],
    'min_samples_leaf': [1],
    'max_features': ['log2'],
    'bootstrap': [True]

}


rf_clf = RandomForestClassifier(random_state=50)

random_search = RandomizedSearchCV(
    estimator=rf_clf,
    param_distributions=param_dist,
    n_iter=8, 
    scoring='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=1,
    random_state=50
)


random_search.fit(x_train_cleaned, y_train)

best_rf_clf = random_search.best_estimator_
y_pred = best_rf_clf.predict(x_test_cleaned)


print(f'Лучшие параметры: {random_search.best_params_}')
print(f'Средняя точность (CV): {random_search.best_score_:.4f}')
print(f'Точность на тестовых данных: {accuracy_score(y_test, y_pred):.4f}')
print('Матрица ошибок:')
print(confusion_matrix(y_test, y_pred))

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Лучшие параметры: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'bootstrap': True}
Средняя точность (CV): 0.7535
Точность на тестовых данных: 0.7675
Матрица ошибок:
[[797  44 105]
 [ 10 825 140]
 [141 231 593]]


4. Анализ влияния фичей на модель

До этого в задачах вы работали над подготовленным датасетом, где фичи были заранее извлечены из текстовых переменных, отскейлены и пропущены через OneHotEncoder. Сравним, какой была бы предсказательная способность модели, если бы мы использовали только сырые данные из исходного датасета. Для этого проделайте следующие шаги:

1. Загрузите датасет `vehicles_dataset_old.csv`.
2. Удалите из него переменную `price` и все строковые колонки. Дерево решений и случайный лес не умеют самостоятельно работать со строковыми значениями.
3. Сформируйте x_train и x_test так же, как они были сформированы в предыдущих заданиях.
4. Обучите свою лучшую модель случайного леса на новых данных и замерьте качество. Убедитесь, что оно ухудшилось.
5. Найдите три фичи, которые лучшим образом повлияли на предсказательную способность модели.

In [91]:
df_old = pd.read_csv('data/vehicles_dataset_old.csv')
df_old.head()

,id,url,region,region_url,price,year,manufacturer,model,fuel,odometer,title_status,transmission,image_url,description,state,lat,long,posting_date,price_category,date
0,7308295377,https://chattanooga.craigslist.org/ctd/d/chatt...,chattanooga,https://chattanooga.craigslist.org,54990,2020,ram,2500 crew cab big horn,diesel,27442,clean,other,https://images.craigslist.org/00N0N_1xMPvfxRAI...,Carvana is the safer way to buy a car During t...,tn,35.060000,-85.250000,2021-04-17T12:30:50-0400,high,2021-04-17 16:30:50+00:00
1,7316380095,https://newjersey.craigslist.org/ctd/d/carlsta...,north jersey,https://newjersey.craigslist.org,16942,2016,ford,explorer 4wd 4dr xlt,other,60023,clean,automatic,https://images.craigslist.org/00x0x_26jl9F0cnL...,***Call Us for more information at: 201-635-14...,nj,40.821805,-74.061962,2021-05-03T15:40:21-0400,medium,2021-05-03 19:40:21+00:00
2,7313733749,https://reno.craigslist.org/ctd/d/atlanta-2017...,reno / tahoe,https://reno.craigslist.org,35590,2017,volkswagen,golf r hatchback,gas,14048,clean,other,https://images.craigslist.org/00y0y_eeZjWeiSfb...,Carvana is the safer way to buy a car During t...,ca,33.779214,-84.411811,2021-04-28T03:52:20-0700,high,2021-04-28 10:52:20+00:00
3,7308210929,https://fayetteville.craigslist.org/ctd/d/rale...,fayetteville,https://fayetteville.craigslist.org,14500,2013,toyota,rav4,gas,117291,clean,automatic,https://images.craigslist.org/00606_iGe5iXidib...,2013 Toyota RAV4 XLE 4dr SUV Offered by: R...,nc,35.715954,-78.655304,2021-04-17T10:08:57-0400,medium,2021-04-17 14:08:57+00:00
4,7303797340,https://knoxville.craigslist.org/ctd/d/knoxvil...,knoxville,https://knoxville.craigslist.org,14590,2012,bmw,1 series 128i coupe 2d,other,80465,clean,other,https://images.craigslist.org/00F0F_5UAXmOzC18...,Carvana is the safer way to buy a car During t...,tn,35.970000,-83.940000,2021-04-08T15:10:56-0400,medium,2021-04-08 19:10:56+00:00


In [93]:
# Ваш код здесь 
y = df_old['price_category']
x = df_old.drop(['price', 'price_category'], axis=1).select_dtypes(exclude=['object'])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

rf_old = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    max_features='log2',
    min_samples_leaf=1,
    min_samples_split=2,
    bootstrap=True,
    random_state=42
)

rf_old.fit(x_train, y_train)
y_pred_old = rf_old.predict(x_test)

print(f'Accuracy on old dataset: {accuracy_score(y_test, y_pred_old):.4f}')
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_old))

Accuracy on old dataset: 0.6403
Confusion matrix:
[[713  73 211]
 [ 48 709 196]
 [274 236 426]]


In [94]:
feat_imp = list(zip(x_train.columns, rf_old.feature_importances_))
feat_imp_sorted = sorted(feat_imp, key=lambda x: x[1], reverse=True)

print('Top 3 features:')
for feature, importance in feat_imp_sorted[:3]:
    print(f"{feature:20s} {importance:.4f}")

Top 3 features:
odometer             0.2722
year                 0.2300
id                   0.1680
